# Stage 06a — Diagnose the 2022 validation candidate

This diagnostic consumes an existing solved network; it does not run the full-year solve. It reports dispatch, potential, curtailment, capacity factors, and monthly carrier totals. Reservoir/cascade physics and threshold changes are intentionally out of scope.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.artifacts import run_preflight, format_preflight_report
from src.phase06b import aggregate_dispatch_by_carrier
import pypsa
result_dir = ROOT / 'results/phase_06a'
result_dir.mkdir(parents=True, exist_ok=True)
candidate = ROOT / 'results/phase_06/network_solved_2022.nc'
if not candidate.exists(): candidate = ROOT / 'data/processed/networks/network_base_filled.nc'
network = pypsa.Network(candidate)
summary = aggregate_dispatch_by_carrier(network)
summary.to_csv(result_dir / 'dispatch_by_carrier.csv', index=False)
monthly = network.generators_t.p.groupby(network.snapshots.month).sum().T
monthly.to_csv(result_dir / 'dispatch_by_generator_month.csv')
diagnostic = {'candidate': str(candidate.relative_to(ROOT)), 'snapshots': len(network.snapshots), 'status': 'diagnostic_only', 'reservoir_refinement': 'deferred', 'thresholds_changed': False}
(result_dir / 'diagnostic_metadata.json').write_text(json.dumps(diagnostic, indent=2), encoding='utf-8')
display(summary)

## Interpretation

Use this output to separate temporal/profile curtailment from network effects. The accounting bridge remains explicit and unresolved residuals are not modeled as losses.